In [ ]:
from pathlib import Path
import sys

# Resolve the project from the notebook's working directory (root or any subfolder).
start = Path.cwd().resolve()
PROJECT_ROOT = next(
    (folder for folder in (start, *start.parents)
     if (folder / "replication_paths.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Start Jupyter from the replication project or a subfolder of it. "
        "The project root must contain replication_paths.py."
    )
for source_dir in (PROJECT_ROOT, PROJECT_ROOT / "01_taxonomy", PROJECT_ROOT / "03_judge_prompts",
                   PROJECT_ROOT / "04_llm_evaluation"):
    if str(source_dir) not in sys.path:
        sys.path.insert(0, str(source_dir))
from replication_paths import DATASET_DIR, RAW_DEVGPT_DIR, INTERIM_DIR, RESULTS_DIR, HUMAN_FILE, OUTPUT_DIR

import pandas as pd

PROCESSED_DIR = DATASET_DIR
INPUT_PATH = PROCESSED_DIR / "devgpt_first_turn_english.parquet"
PARQUET_OUTPUT = PROCESSED_DIR / "devgpt_llm_judge_sample_260.parquet"
CSV_OUTPUT = PROCESSED_DIR / "devgpt_llm_judge_sample_260.csv"
SAMPLE_SIZE = 260
RANDOM_STATE = 42
if not INPUT_PATH.is_file():
    raise FileNotFoundError(
        f"Missing preprocessed DevGPT file: {INPUT_PATH}. "
        "Run 01_extract_and_filter_clean.ipynb after obtaining the raw snapshots, "
        "or use the included 260-prompt sample directly."
    )

In [ ]:
df = pd.read_parquet(INPUT_PATH)

print(f"Dataset loaded: {len(df):,} prompts")
print("Columns:", df.columns.tolist())

In [ ]:
before = len(df)

df_sampling_frame = (
    df.drop_duplicates(
        subset=["prompt"],
        keep="first"
    )
    .reset_index(drop=True)
    .copy()
)

removed = before - len(df_sampling_frame)

print(f"Rows before prompt-text deduplication: {before:,}")
print(f"Exact duplicate prompt texts removed: {removed:,}")
print(f"Sampling-frame rows: {len(df_sampling_frame):,}")
print(f"Duplicate prompts remaining: {df_sampling_frame['prompt'].duplicated().sum():,}")
print(f"Duplicate URLs remaining: {df_sampling_frame['chatgpt_url'].duplicated().sum():,}")

In [ ]:
if len(df_sampling_frame) < SAMPLE_SIZE:
    raise ValueError(
        f"Sampling frame has only {len(df_sampling_frame):,} rows, "
        f"which is smaller than the requested sample of {SAMPLE_SIZE:,}."
    )

df_sample = (
    df_sampling_frame.sample(
        n=SAMPLE_SIZE,
        random_state=RANDOM_STATE,
        replace=False
    )
    .reset_index(drop=True)
    .copy()
)

print(f"Sample size: {len(df_sample):,}")

In [ ]:
df_sample.insert(
    0,
    "prompt_id",
    [f"P{i:04d}" for i in range(1, len(df_sample) + 1)]
)

print(
    df_sample[
        ["prompt_id", "chatgpt_url", "prompt"]
    ].head()
)

In [ ]:
df_sample.to_parquet(
    PARQUET_OUTPUT,
    index=False
)

df_sample.to_csv(
    CSV_OUTPUT,
    index=False
)

print(f"Parquet saved: {PARQUET_OUTPUT}")
print(f"CSV saved: {CSV_OUTPUT}")